In [2]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

from matplotlib import cm
from matplotlib.ticker import LinearLocator
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal

In [71]:
def do_fit(X, Y, raw_data, field):
    rawer_data = np.concatenate([np.repeat([[x,y]],z, axis=0) for x, y, z in zip(raw_data.H, raw_data.L, raw_data.loc[slice(None), field])])
    pos = np.dstack((X, Y))
    covar = np.cov(rawer_data.T)
    means = np.mean(rawer_data, axis=0)
    rv = multivariate_normal(mean=means.T, cov=covar.T)
    fitz = rv.pdf(pos)
    fitz = fitz/np.sum(fitz)

    return fitz


In [72]:
def get_field(data, Hs, Ls, field):
    vals = []
    for H in Hs:
        toadd = []
        for L in Ls:
            mask = np.logical_and(data.H==H, data.L==L)
            toadd.append(data.loc[mask, field])
        vals.append(toadd)
    return vals

In [ ]:
def get_grid(data, field):
    Ys = data.loc[slice(None),field]
    Ys=Ys/np.sum(Ys)
    Hs = data.loc[slice(None), 'H'].unique()
    Ls = data.loc[slice(None), 'L'].unique()
    
    Z = np.array(get_field(data, Hs, Ls, field))
    X, Y = np.meshgrid(Hs, Ls)
    Z = Z/np.sum(Z)

    return X, Y, Z

In [73]:
def get_zs(data, field):
    normz = (Z/np.sum(Z)).T[0]
    fitz = do_fit(X, Y, data, field)
    diff = np.array(fitz)-np.array(normz)

    return normz, fitz, diff

In [74]:
raw_data = pd.read_csv("all_humsaver_enrich_txt_h.csv", names=["L","H","dSNPs", "nSNPs", "SNPs", "Enrich", "p"], skiprows=1)

In [75]:
normz, fitz, diff = get_zs(raw_data, 'SNPs')

In [79]:
%matplotlib
fig, [normax, diffax, fitax] = plt.subplots(1, 3)
themin = 0
themax = np.max([normz, diff, fitz])


diffax.imshow(diff.T, vmin=themin, vmax=themax, origin='lower')
normax.imshow(normz.T, vmin=themin, vmax=themax, origin='lower')
fitax.imshow(fitz.T, vmin=themin, vmax=themax, origin='lower')

Using matplotlib backend: MacOSX


: 

In [77]:
%matplotlib
histgrid = [[raw_data.loc[np.logical_and(raw_data.H==H,raw_data.L==L), 'dSNPs'] for L in Ls] for H in Hs]
histgrid = np.cumsum(histgrid, axis=1)
histgrid = np.cumsum(histgrid, axis=0)

plt.imshow(histgrid, origin='lower')

Using matplotlib backend: MacOSX


In [45]:
%matplotlib
fig, ax = plt.subplots(subplot_kw={"projection": "3d"})

surf = ax.scatter3D(X, Y, Z.T, c=Z.T,cmap=cm.coolwarm, linewidth=0, antialiased=False)
ax.zaxis.set_major_locator(LinearLocator(10))
ax.zaxis.set_major_formatter('{x:.02f}')
fig.colorbar(surf, shrink=0.5, aspect=5)

plt.show()

Using matplotlib backend: MacOSX
